# Pythia-1B: cosine routing + neural reranking

PPL проверяется только на native-контексте 2048. На 14K, 32K и 100K измеряется только скорость. Варианты: dense, full-scan cosine, hierarchical cosine, cosine + neural reranker и learned selector без cosine-фильтра.

In [ ]:
from pathlib import Path
import gc
import importlib
import json
import math
import time

import torch
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from ..backend.model import (
    DEVICE,
    DTYPE,
    PythiaConfig,
    ROUTING_CONFIG,
    clear_gpu_cache,
)
from ..backend import routing_ablation_benchmark
importlib.reload(routing_ablation_benchmark)
from routing_ablation_benchmark import (
    collect_reranker_teacher_samples,
    dense_ppl,
    load_base_model,
    load_reranker,
    report_needle,
    report_quality,
    report_speed,
    repeat_ids,
)

MODEL_DIR = Path(
    "/home/froschin/.cache/huggingface/hub/models--EleutherAI--pythia-1b/"
    "snapshots/f73d7dcc545c8bd326d8559c8ef84ffe92fea6b2"
)
RERANKER_CHECKPOINT = Path("./checkpoints/block-reranker-v3.pt")
TEXT_FILE = Path("/home/froschin/work/llm/tinyshakespeare.txt")
OUTPUT_FILE = Path("./pythia_1b_routing_results.json")
CHUNK_SIZE = 256
NEW_TOKENS = 16
QUALITY_CONTEXT = 2048
SPEED_CONTEXTS = (2048, 14000, 32768, 100000)
ROUTED_VARIANTS = (
    "full_scan_cosine",
    "hierarchical_cosine",
    "full_scan_reranker",
    "hierarchical_reranker",
    "neural_full_scan",
)
print({"device": str(DEVICE), "dtype": str(DTYPE), "routing": ROUTING_CONFIG})

/home/froschin/work/llm/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'device': 'cuda', 'dtype': 'torch.float16', 'routing': {'block_size': 256, 'route_blocks': 16, 'beam_width': 32, 'summary_parts': 4, 'global_blocks': 1, 'local_blocks': 2, 'local_window': 256, 'route_refresh_interval': 64}}


In [2]:
if not MODEL_DIR.exists():
    MODEL_DIR = Path(
        snapshot_download(
            repo_id="EleutherAI/pythia-1b",
            allow_patterns=["config.json", "tokenizer*", "*.json", "*.safetensors", "*.bin"],
        )
    )

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True)
filler_ids = torch.tensor(
    tokenizer(TEXT_FILE.read_text(encoding="utf-8"), add_special_tokens=False).input_ids,
    dtype=torch.long,
)
reranker = load_reranker(RERANKER_CHECKPOINT, PythiaConfig().head_dim)
print({
    "model_dir": str(MODEL_DIR),
    "filler_tokens": filler_ids.numel(),
    "reranker_checkpoint": str(RERANKER_CHECKPOINT),
})

loaded reranker: checkpoints/block-reranker-v2.pt
{'model_dir': '/home/froschin/.cache/huggingface/hub/models--EleutherAI--pythia-1b/snapshots/f73d7dcc545c8bd326d8559c8ef84ffe92fea6b2', 'filler_tokens': 340240, 'reranker_checkpoint': 'checkpoints/block-reranker-v2.pt'}


## 1. PPL на 2048

Длинные контексты не используются для PPL в этом протоколе: Pythia была обучена на 2048, поэтому 14K+ оцениваются только по скорости.

In [3]:
quality_rows = report_quality(
    MODEL_DIR,
    filler_ids,
    reranker,
    contexts=(QUALITY_CONTEXT,),
    chunk_size=CHUNK_SIZE,
    modes=("dense",) + ROUTED_VARIANTS,
)
dense_quality = next(row for row in quality_rows if row["routing"] == "dense")
for row in quality_rows:
    if "perplexity" in row and row["routing"] != "dense":
        row["ppl_delta_vs_dense"] = row["perplexity"] - dense_quality["perplexity"]
        row["ppl_relative_percent_vs_dense"] = (
            100.0 * row["ppl_delta_vs_dense"] / dense_quality["perplexity"]
        )
print(json.dumps(quality_rows, indent=2))

ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
[
  {
    "routing": "dense",
    "context_length": 2048,
    "tokens": 2047,
    "mean_nll": 3.0692601203918457,
    "perplexity": 21.525970458984375,
    "seconds": 0.2330290551763028
  },
  {
    "routing": "full_scan_cosine",
    "context_length": 2048,
    "tokens": 2047,
    "mean_nll": 3.0780513286590576,
    "perplexity": 21.71604347229004,
    "routing_seconds": 0.12097607459872961,
    "attention_kernel_seconds": 0.0382369770668447,
    "route_calls_all_layers": 96,
    "mean_candidates_per_route": 2.5,
    "ppl_delta_vs_dense": 0.19007301330566406,
    "ppl_relative_percent_vs_dense": 0.8829939336199942
  },
  {
    "routing": "hierarchical_cosine",
    "context_length": 2048,
    "tokens": 2047,
    "mean_nll": 3.0780513286590576,
    "perplexity": 2

## 2. Speed на длинном контексте

Dense запускается только на 2048. Routed-варианты запускаются на 2048, 14K, 32K и 100K. Сохраняются routing latency, attention-kernel latency, prefill/decode throughput и peak memory.

In [4]:
speed_rows = report_speed(
    MODEL_DIR,
    filler_ids,
    reranker,
    contexts=SPEED_CONTEXTS,
    chunk_size=CHUNK_SIZE,
    new_tokens=NEW_TOKENS,
    modes=("dense",) + ROUTED_VARIANTS,
)
dense_speed = next(row for row in speed_rows if row["routing"] == "dense")
for row in speed_rows:
    if row["routing"] != "dense" and row.get("prompt_length") == 2048:
        row["decode_speedup_vs_dense"] = (
            dense_speed["decode_tokens_per_second"] / row["decode_tokens_per_second"]
        )
        row["total_speedup_vs_dense"] = (
            dense_speed["total_seconds"] / row["total_seconds"]
        )
print(json.dumps(speed_rows, indent=2))

ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
[
  {
    "routing": "dense",
    "prompt_length": 2048,
    "new_tokens": 16,
    "prefill_seconds": 0.08074829401448369,
    "prefill_tokens_per_second": 25362.764935103813,
    "decode_seconds": 0.21245892206206918,
    "decode_tokens_per_second": 75.30867541220816,
    "total_seconds": 0.29320721607655287,
    "peak_cuda_allocated_gib": 2.835397243499756,
    "peak_cuda_reserved_gib": 2.978515625
  },
  {
    "prompt_length": 2048,
    "chunk_size": 256,
    "new_tokens": 16,
    "prefill_seconds": 0.3682642311323434,
    "prefill_tokens_per_second": 5561.224324455254,
    "decode_seconds": 0.533032963052392,
    "decode_tokens_per_second": 30.01690534929892,
    "total_seconds": 0.9012971941847354,
    "routing": "full_scan_cosine",
    "routing_seconds": 0

## 3. Needle retrieval на 2048

In [5]:
needle_rows = report_needle(
    MODEL_DIR,
    tokenizer,
    filler_ids,
    reranker,
    contexts=(QUALITY_CONTEXT,),
    chunk_size=CHUNK_SIZE,
    modes=("dense",) + ROUTED_VARIANTS,
)
print(json.dumps(needle_rows, indent=2))

ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
ignored auxiliary checkpoint keys: 48
[
  {
    "routing": "dense",
    "prompt_length": 2048,
    "needle_position": 512,
    "answer_mean_nll": 0.49021458625793457,
    "answer_perplexity": 1.6326664686203003,
    "text_exact_match": false,
    "predicted_answer": "\nBIT-314159",
    "target_answer": " ORBIT-314159"
  },
  {
    "prompt_length": 2048,
    "needle_position": 512,
    "answer_mean_nll": 0.6333163976669312,
    "answer_perplexity": 1.8838478326797485,
    "text_exact_match": false,
    "predicted_answer": "\nBIT-314159",
    "target_answer": " ORBIT-314159",
    "routing": "full_scan_cosine"
  },
  {
    "prompt_length": 2048,
    "needle_position": 512,
    "answer_mean_nll": 0.607815682888031,
    "answer_perplexity": 1.8364157676696777,
    "text_exact_match": false,
    "predicte

## 4. Recall@64 и Recall@16

При production block_size=256 контекст 2048 содержит только 8 блоков, поэтому Recall@64 был бы тривиальным. Для содержательного selector-диагностического теста ниже используются summaries по 16 токенов: 2048 токенов дают 128 диагностических блоков. Это не меняет production PPL/speed конфигурацию.

Oracle — Top-16 блоков по dense attention mass. Сравниваются cosine Top-64, hierarchical Top-64, cosine-only Top-16, reranker после cosine Top-16 и neural selector Top-16 без cosine-фильтра.

In [6]:
def hierarchical_topk(query, summaries, candidate_k=64, beam_width=32):
    count = summaries.shape[0]
    capacity = 1
    while capacity < count:
        capacity *= 2
    tree_sum = torch.zeros(
        2 * capacity - 1,
        summaries.shape[-1],
        device=summaries.device,
    )
    tree_count = torch.zeros(2 * capacity - 1, device=summaries.device)
    leaf_start = capacity - 1
    tree_sum[leaf_start:leaf_start + count] = summaries.float()
    tree_count[leaf_start:leaf_start + count] = 1.0
    for node in range(leaf_start - 1, -1, -1):
        left = 2 * node + 1
        right = left + 1
        tree_sum[node] = tree_sum[left] + tree_sum[right]
        tree_count[node] = tree_count[left] + tree_count[right]

    query = F.normalize(query.float(), dim=-1)
    candidates = torch.tensor([0], device=summaries.device, dtype=torch.long)
    for _ in range(int(math.log2(capacity))):
        children = torch.cat((2 * candidates + 1, 2 * candidates + 2))
        valid = tree_count[children] > 0
        child_summary = tree_sum[children] / tree_count[children].clamp_min(1).unsqueeze(-1)
        scores = (F.normalize(child_summary.float(), dim=-1) * query).sum(-1)
        scores = scores.masked_fill(~valid, float("-inf"))
        keep = min(beam_width, int(valid.sum()))
        candidates = children[torch.topk(scores, keep).indices]

    leaves = (candidates - leaf_start).clamp(0, count - 1)
    scores = (
        F.normalize(summaries[leaves].float(), dim=-1) * query
    ).sum(-1)
    keep = min(candidate_k, leaves.numel())
    return leaves[torch.topk(scores, keep).indices]


def overlap_recall(selected, oracle):
    return len(set(selected.tolist()) & set(oracle.tolist())) / max(1, len(oracle))

In [7]:
RECALL_BLOCK_SIZE = 16
RECALL_SEQUENCES = 2
RECALL_QUERY_STRIDE = 128
RECALL_MAX_SAMPLES = 64

teacher_model = load_base_model(MODEL_DIR)
teacher_sequences = [
    repeat_ids(filler_ids, 2048)
    for _ in range(RECALL_SEQUENCES)
]
queries, summaries, value_summaries, positions, target_mass, valid = collect_reranker_teacher_samples(
    teacher_model,
    teacher_sequences,
    block_size=RECALL_BLOCK_SIZE,
    query_stride=RECALL_QUERY_STRIDE,
    summary_parts=ROUTING_CONFIG["summary_parts"],
    return_multiscale=True,
)
del teacher_model
clear_gpu_cache()

rows = []
rerank_times_ms = []
for sample in range(min(RECALL_MAX_SAMPLES, queries.shape[0])):
    for head in range(queries.shape[1]):
        block_count = int(valid[sample, head].sum())
        if block_count < 64:
            continue

        query = queries[sample, head].to(DEVICE)
        block_summary = summaries[sample, head, :block_count].to(DEVICE)
        block_value_summary = value_summaries[sample, head, :block_count].to(DEVICE)
        block_position = positions[sample, :block_count].to(DEVICE)
        block_summary_mean = block_summary.mean(dim=-2)
        target = target_mass[sample, head, :block_count]
        oracle = torch.topk(target, k=16).indices

        cosine_scores = (
            F.normalize(query.float(), dim=-1)
            * F.normalize(block_summary_mean.float(), dim=-1)
        ).sum(-1)
        cosine64 = torch.topk(cosine_scores, k=64).indices
        cosine16 = torch.topk(cosine_scores, k=16).indices

        started = time.perf_counter()
        neural_scores = reranker(
            query.view(1, -1),
            block_summary.view(1, block_count, ROUTING_CONFIG["summary_parts"], -1),
            block_value_summary.view(1, block_count, ROUTING_CONFIG["summary_parts"], -1),
            block_position.view(1, block_count),
        )[0]
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        rerank_times_ms.append((time.perf_counter() - started) * 1000.0)

        neural_full16 = torch.topk(neural_scores, k=16).indices
        candidate_summary = block_summary[cosine64]
        candidate_value_summary = block_value_summary[cosine64]
        candidate_position = block_position[cosine64]
        candidate_scores = reranker(
            query.view(1, -1),
            candidate_summary.view(1, 64, ROUTING_CONFIG["summary_parts"], -1),
            candidate_value_summary.view(1, 64, ROUTING_CONFIG["summary_parts"], -1),
            candidate_position.view(1, 64),
        )[0]
        rerank16 = cosine64[torch.topk(candidate_scores, k=16).indices]

        hierarchical64 = hierarchical_topk(
            query,
            block_summary_mean,
            candidate_k=64,
            beam_width=64,
        )
        hierarchical_scores = reranker(
            query.view(1, -1),
            block_summary[hierarchical64].view(1, 64, ROUTING_CONFIG["summary_parts"], -1),
            block_value_summary[hierarchical64].view(1, 64, ROUTING_CONFIG["summary_parts"], -1),
            block_position[hierarchical64].view(1, 64),
        )[0]
        hierarchical16 = hierarchical64[
            torch.topk(hierarchical_scores, k=16).indices
        ]

        rows.append({
            "cosine_recall_at_64": overlap_recall(cosine64, oracle),
            "hierarchical_recall_at_64": overlap_recall(hierarchical64, oracle),
            "cosine_only_recall_at_16": overlap_recall(cosine16, oracle),
            "reranker_after_cosine_recall_at_16": overlap_recall(rerank16, oracle),
            "neural_full_scan_recall_at_16": overlap_recall(neural_full16, oracle),
            "hierarchical_reranker_recall_at_16": overlap_recall(
                hierarchical16,
                oracle,
            ),
        })

recall_summary = {
    "oracle": "dense attention mass Top-16",
    "diagnostic_block_size": RECALL_BLOCK_SIZE,
    "samples": len(rows),
    "mean_reranker_latency_ms_per_head": sum(rerank_times_ms) / max(1, len(rerank_times_ms)),
    **{
        key: sum(row[key] for row in rows) / max(1, len(rows))
        for key in rows[0]
    },
}
print(json.dumps(recall_summary, indent=2))

ignored auxiliary checkpoint keys: 48
{
  "oracle": "dense attention mass Top-16",
  "diagnostic_block_size": 16,
  "samples": 256,
  "mean_reranker_latency_ms_per_head": 0.6341972939480911,
  "cosine_recall_at_64": 0.995361328125,
  "hierarchical_recall_at_64": 0.995361328125,
  "cosine_only_recall_at_16": 0.8681640625,
  "reranker_after_cosine_recall_at_16": 0.685302734375,
  "neural_full_scan_recall_at_16": 0.677978515625,
  "hierarchical_reranker_recall_at_16": 0.685302734375
}


## Results

In [8]:
results = {
    "protocol": {
        "quality_context": QUALITY_CONTEXT,
        "speed_contexts": SPEED_CONTEXTS,
        "production_routing": ROUTING_CONFIG,
        "recall_diagnostic_block_size": RECALL_BLOCK_SIZE,
        "variants": ("dense",) + ROUTED_VARIANTS,
    },
    "quality_2048": quality_rows,
    "speed": speed_rows,
    "needle_2048": needle_rows,
    "recall_2048": recall_summary,
}
OUTPUT_FILE.write_text(
    json.dumps(results, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("saved:", OUTPUT_FILE)

saved: pythia_1b_routing_results.json
